In [ ]:
"""
KoChatGPT 데이터셋 EDA 워크북
========================================

【 대상 데이터 】
- kochatgpt_1_PPO.jsonl (PPO 학습용)

PPO 데이터의 특징:
- Prompt만 있음 (Completion은 없음)
- SFT에서 본 질문들을 다시 제시해서 모델이 더 나은 답변을 생성하도록 학습
- SFT prompt와 100% 중복이 정상이고 의도된 구성

분석 목표:
1. Prompt 기본 통계
2. 빈 Prompt 탐지 (제거 필수)
3. 파싱 에러 탐지 (정제 필수)
4. PPO 내부 중복 탐지 (중복 제거)
5. 정제 후 데이터 품질 확인
"""

In [1]:
# === Colab 호환 셋업 (자동 추가) ===
import torch as _t
_t._orig_load = getattr(_t, '_orig_load', _t.load)
def _compat_load(*a, **k):
    k.setdefault('weights_only', False)
    return _t._orig_load(*a, **k)
_t.load = _compat_load
try:
    import matplotlib; matplotlib.rcParams['axes.unicode_minus'] = False
except Exception: pass
print('[colab-compat] torch.load weights_only=False 패치')


[colab-compat] torch.load weights_only=False 패치


In [2]:
!git clone https://github.com/airobotlab/KoChatGPT
!cp -r /content/KoChatGPT/colossalai_ChatGPT_230319/chatgpt /content/chatgpt

Cloning into 'KoChatGPT'...
remote: Enumerating objects: 304, done.
remote: Total 304 (delta 0), reused 0 (delta 0), pack-reused 304 (from 1)
Receiving objects: 100% (304/304), 57.72 MiB | 28.52 MiB/s, done.
Resolving deltas: 100% (123/123), done.


In [4]:
"""
KoChatGPT PPO (Proximal Policy Optimization) 데이터셋 EDA
=========================================================

PPO 데이터의 특징:
- Prompt만 있음 (Completion은 없음)
- SFT에서 본 질문들을 다시 제시해서 모델이 더 나은 답변을 생성하도록 학습
- SFT prompt와 100% 중복이 정상이고 의도된 구성

분석 목표:
1. Prompt 기본 통계
2. 빈 Prompt 탐지 (제거 필수)
3. 파싱 에러 탐지 (정제 필수)
4. PPO 내부 중복 탐지 (중복 제거)
5. 정제 후 데이터 품질 확인
"""

import json
from collections import Counter
import statistics
import re

# ============================================================================
# 데이터 로드
# ============================================================================

print("="*80)
print("PPO (강화학습) 데이터 EDA & 정제 가이드")
print("="*80)

print("\n【 1단계: 데이터 로드 】\n")

try:
    with open('/content/KoChatGPT/data_kochatgpt/kochatgpt_3_PPO.jsonl', 'r', encoding='utf-8') as f:
        ppo_data = json.load(f)
    print(f"✓ kochatgpt_1_PPO.jsonl 로드 성공")
    print(f"  총 {len(ppo_data)}건의 레코드\n")
except FileNotFoundError:
    print("✗ 파일을 찾을 수 없음: kochatgpt_1_PPO.jsonl")
    ppo_data = []
except json.JSONDecodeError as e:
    print(f"✗ JSON 파싱 오류: {e}")
    ppo_data = []

if not ppo_data:
    print("데이터가 없어서 분석 불가능")
    exit()

# ============================================================================
# 기본 통계
# ============================================================================

print("="*80)
print("【 2단계: 기본 통계 】\n")

prompts = [r.get('prompt', '') for r in ppo_data if 'prompt' in r]
prompt_lengths = [len(p) for p in prompts]

print(f"총 레코드: {len(ppo_data)}건")
print(f"Prompt 개수: {len(prompts)}건")
print(f"평균 길이: {statistics.mean(prompt_lengths):.1f}자")
print(f"길이 범위: {min(prompt_lengths)} ~ {max(prompt_lengths)}자\n")

# ============================================================================
# 정제 대상 1: 빈 Prompt
# ============================================================================

print("="*80)
print("【 3단계: 빈 Prompt 탐지 】\n")

empty_prompts = [p for p in prompts if not p or not p.strip()]
print(f"빈 또는 공백만 있는 Prompt: {len(empty_prompts)}건 ({len(empty_prompts)/len(ppo_data)*100:.1f}%)")

if empty_prompts:
    print("\n샘플:")
    for i, p in enumerate(empty_prompts[:5], 1):
        print(f"  {i}. {repr(p)}")

# ============================================================================
# 정제 대상 2: 파싱 에러
# ============================================================================

print("="*80)
print("【 4단계: 파싱 에러 탐지 】\n")

parsing_errors = []
for idx, p in enumerate(prompts):
    # dict 파싱 실패 지표
    if 'token' in p and ('}' in p[-20:] or '"' in p[-20:]):
        parsing_errors.append((idx, p))

print(f"파싱 에러 ('token'/'}}' 포함): {len(parsing_errors)}건 ({len(parsing_errors)/len(ppo_data)*100:.1f}%)")

if parsing_errors:
    print("\n샘플 (처음 5개):")
    for i, (idx, p) in enumerate(parsing_errors[:5], 1):
        print(f"  {i}. {p[-100:]}...")

# ============================================================================
# 정제 대상 3: PPO 내부 중복
# ============================================================================

print("="*80)
print("【 5단계: PPO 내부 중복 탐지 】\n")

prompt_counter = Counter(prompts)
duplicates = {p: count for p, count in prompt_counter.items() if count > 1}

print(f"완전 중복 Prompt: {len(duplicates)}가지")
total_duplicate_count = sum(count - 1 for count in duplicates.values())
print(f"제거 대상 (중복): {total_duplicate_count}건 ({total_duplicate_count/len(ppo_data)*100:.1f}%)\n")

if duplicates:
    print("중복 빈도 (상위 10):")
    for prompt, count in sorted(duplicates.items(), key=lambda x: -x[1])[:10]:
        print(f"  [{count}회] {prompt[:80]}")

# ============================================================================
# 정제 후 예상 결과
# ============================================================================

print("="*80)
print("【 6단계: 정제 후 결과 예상 】\n")

removal_count = len(empty_prompts) + len(parsing_errors) + total_duplicate_count
final_count = len(ppo_data) - removal_count

print(f"정제 전: {len(ppo_data):,}건")
print(f"  • 빈 Prompt: -{len(empty_prompts)}건")
print(f"  • 파싱 에러: -{len(parsing_errors)}건")
print(f"  • 내부 중복: -{total_duplicate_count}건")
print(f"정제 후: {final_count:,}건")
print(f"보존율: {final_count/len(ppo_data)*100:.1f}%\n")

# ============================================================================
# 샘플 데이터
# ============================================================================

print("="*80)
print("【 데이터 샘플 】\n")

print("처음 5개 Prompt:")
for i, p in enumerate(prompts[:5], 1):
    print(f"{i}. ({len(p)}자) {p}")

print("\n가장 짧은 Prompt:")
for i, p in enumerate(sorted(prompts, key=len)[:3], 1):
    print(f"{i}. ({len(p)}자) {repr(p)}")

print("\n가장 긴 Prompt:")
for i, p in enumerate(sorted(prompts, key=len, reverse=True)[:3], 1):
    print(f"{i}. ({len(p)}자) {p[:80]}...")

PPO (강화학습) 데이터 EDA & 정제 가이드

【 1단계: 데이터 로드 】

✓ kochatgpt_1_PPO.jsonl 로드 성공
  총 12000건의 레코드

【 2단계: 기본 통계 】

총 레코드: 12000건
Prompt 개수: 12000건
평균 길이: 22.2자
길이 범위: 0 ~ 295자

【 3단계: 빈 Prompt 탐지 】

빈 또는 공백만 있는 Prompt: 3건 (0.0%)

샘플:
  1. ''
  2. ''
  3. ''
【 4단계: 파싱 에러 탐지 】

파싱 에러 ('token'/'}' 포함): 12건 (0.1%)

샘플 (처음 5개):
  1. 으로도 바뀌지는 않을 것 같다'는 입장을 표명했다는 것은 그가 그동안 말한 어떤 단일한 인터뷰보다도 보다 일반적인 생각이라고 할 수 있을 것 같습니다.", 'token': 257}...
  2. 비극을 다룬다. 또한 작가는 사망한 소녀의 죽음과 함께 헝가리 전쟁과 피난민 문제를 생각하게 하며, 인간의 삶과 사회적 현실을 진지하게 생각하게 한다.", 'token': 233}...
  3. 'DNA'의 빌보드 핫 100 진입 순위는?", 'completion': "방탄소년단의 곡 'DNA'는 빌보드 핫 100에서 최고 67위를 기록했습니다.", 'token': 79}...
  4.  어시스턴트이기 때문에 '빅 대디'가 무엇을 입고 있는지 파악할 수 없습니다. 해당 정보를 제공해주실 수 있으면 더욱 정확한 답변을 드릴 수 있습니다.", 'token': 102}...
  5. ompletion': "SF단편영화 'Episode 1 Fragile : 경계의 저편'의 감독 및 각본은 캄 로메로(Cam Romero)가 맡았습니다.", 'token': 110}...
【 5단계: PPO 내부 중복 탐지 】

완전 중복 Prompt: 45가지
제거 대상 (중복): 54건 (0.4%)

중복 빈도 (상위 10):
  [5회] 얼마에요?
  [4회] 이거는 얼마예요?
  [4회] 영수증 좀 주세요
  [3회] 화

# 【 PPO (Proximal Policy Optimization) 데이터 분석 및 정제 전략】

1. PPO 데이터셋의 특성 및 분석
   - 데이터 구성: Completion 없이 Prompt만 존재함.
   - SFT와의 중복률 (100%):
     • RLHF/PPO 학습 메커니즘 특성상, SFT에서 학습한 동일 질문을 다시 던져
       모델이 더 발전된 고품질 답변을 스스로 생성하도록 유도하는 단계이므로 **100% 중복은 의도된 정상 구조**임.

2. 품질 이슈 현황 및 원인 분석 (총 12,000건 중)
   - Missing / Empty Prompt (3건): Prompt 내용이 비어있는 불량 데이터.
   - Parsing Error / Dict 파싱 이상 (12건): Completion 필드 및 Dict 구문이 Prompt 문자열 내부로 누락·혼입된 데이터.
   - PPO 내부 중복 Prompt (54건 / 45가지): 동일한 프롬프트가 PPO 데이터셋 내부에서 2~5회 중복 등록된 데이터.

3. 항목별 정제 규칙 및 처리 전략
   [1] Empty Prompt (3건) ➔ 【전량 제거】
       • 질문이 존재하지 않아 모델이 답변을 생성할 수 없으므로 제거.

   [2] Parsing Error (12건) ➔ 【정규표현식 전처리 후 질문만 추출 / 이상건 제거】
       • Prompt 뒤에 잘못 붙은 `'completion': ...` 및 token 구문을 정규식(`re.sub`)으로 잘라내어 순수한 질문 텍스트만 복원.
       • 복원이 불가능한 파싱 파손 데이터는 제거.

   [3] PPO 내부 완전 중복 (54건) ➔ 【1건만 유지 후 중복 제거】
       • 같은 프롬프트를 여러 번 학습하여 특정 질문으로 보상이 편향되는 현상을 방지하기 위해
         중복 프롬프트당 1건만 남기고 나머지 제거.

4. 종합 평가 및 정제 후 결과
   - 정제 전: 12,000건
   - 제거/수정 대상: -69건 (빈 Prompt 3건 + 파싱 에러 12건 + 내부 중복 54건)
   - 정제 후 최종 데이터: 11,931건
   - 데이터 보존율: 99.4% (손실률 0.6% 미만으로 안정적으로 강화학습용 프롬프트 Pool 확보)
